![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Give a LangGraph agent live business context with Redis Context Retriever

In [notebook 01](01_intro_context_retriever.ipynb) we turned a data model into auto-generated MCP tools and called them by hand. Now we hand those tools to a **LangGraph ReAct agent** so an LLM can pick and chain them to answer natural-language questions — and we'll see why this is more reliable than a naive vector-RAG lookup for structured data.

<a href="https://colab.research.google.com/github/redis-developer/redis-ai-resources/blob/main/python-recipes/context-retriever/02_agent_with_context_retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **⚠️ Requires Redis Cloud** — a Context Retriever admin key (`CTX_ADMIN_KEY`) and a Redis Cloud database connection (`REDIS_URL`), plus an **OpenAI key** (`OPENAI_API_KEY`). Excluded from CI. See notebook 01 (and this folder's README) for how to get these.

## Setup

In [ ]:
# aiohttp>=3.14.1: openai's aiohttp transport needs it; guards against a stale
# aiohttp preinstalled in the env (e.g. Colab) that pip wouldn't otherwise upgrade.
%pip install -q redis-context-retriever langgraph langchain-openai langchain-core "aiohttp>=3.14.1"

In [2]:
# NBVAL_SKIP
import os, getpass
from urllib.parse import urlparse
for var in ('CTX_ADMIN_KEY', 'REDIS_URL', 'OPENAI_API_KEY'):
    if not os.getenv(var):
        os.environ[var] = getpass.getpass(f'{var}: ')

CTX_ADMIN_KEY = os.environ['CTX_ADMIN_KEY']
CTX_API_URL = os.getenv('CTX_API_URL')
CTX_MCP_URL = os.getenv('CTX_MCP_URL')

# Parse the surface's data source out of REDIS_URL.
_u = urlparse(os.environ['REDIS_URL'])
REDIS_ADDR = f'{_u.hostname}:{_u.port}'
REDIS_USERNAME = _u.username or 'default'
REDIS_PASSWORD = _u.password or ''
REDIS_TLS = _u.scheme == 'rediss'

## Step 1 — A surface to query

We recreate the small e-commerce surface from notebook 01 so this notebook stands alone. (If you kept the surface id and agent key from notebook 01, skip this and set `AGENT_KEY` directly.)

In [3]:
# NBVAL_SKIP
from typing import List
from context_surfaces import (UnifiedClient, export_data_model,
                              ContextModel, ContextField, ContextRelationship,
                              CreateContextSurfaceRequest, DataSourceRequest,
                              DataSourceConnectionConfig)


class Customer(ContextModel):
    __redis_key_template__ = 'customer:{customer_id}'
    customer_id: str = ContextField(description='Unique customer id', is_key_component=True)
    name: str = ContextField(description='Full name', index='text')
    city: str = ContextField(description='City', index='tag')
    lifetime_value: float = ContextField(description='USD spent', index='numeric', sortable=True)
    orders: List['Order'] = ContextRelationship(description='Orders placed', target='Order', source_field='customer_id')


class Order(ContextModel):
    __redis_key_template__ = 'order:{order_id}'
    order_id: str = ContextField(description='Unique order id', is_key_component=True)
    customer_id: str = ContextField(description='Owner customer id', index='tag')
    status: str = ContextField(description='Order status', index='tag')
    total: float = ContextField(description='Order total USD', index='numeric', sortable=True)
    summary: str = ContextField(description='Order description', index='text')


client = UnifiedClient(api_url=CTX_API_URL, mcp_url=CTX_MCP_URL)
await client.__aenter__()

surface = await client._api_client.create_context_surface(
    CreateContextSurfaceRequest(
        name='ecommerce-agent-demo',
        data_model=export_data_model('E-commerce', 'Customers and orders', [Customer, Order]),
        data_source=DataSourceRequest(connection_config=DataSourceConnectionConfig(
            addr=REDIS_ADDR, username=REDIS_USERNAME, password=REDIS_PASSWORD, tls_enabled=REDIS_TLS)),
    ),
    admin_key=CTX_ADMIN_KEY,
)
AGENT_KEY = (await client.create_agent_key(
    admin_key=CTX_ADMIN_KEY, surface_id=surface.id, name='agent')).key

# import_data requires all records in one call to share a model type, so import each entity separately.
for records in (
    [Customer(customer_id='c1', name='Ada Lovelace', city='London', lifetime_value=4200.0),
     Customer(customer_id='c2', name='Alan Turing', city='Manchester', lifetime_value=1500.0)],
    [Order(order_id='o1', customer_id='c1', status='shipped', total=120.0, summary='Keyboard'),
     Order(order_id='o2', customer_id='c1', status='pending', total=650.0, summary='Standing desk'),
     Order(order_id='o3', customer_id='c2', status='shipped', total=80.0, summary='USB-C hub')],
):
    await client.import_data(admin_key=CTX_ADMIN_KEY, context_surface_id=surface.id, records=records)
print('surface ready:', surface.id)

surface ready: 3a9af0ef-36a7-4cc4-b602-450214dbae24


## Step 2 — Wrap the MCP tools as LangChain tools

`list_tools` returns each tool's name, description, and JSON-Schema `inputSchema`. LangChain wants a Python callable plus an `args_schema`, so we convert the JSON Schema into a small Pydantic model and wrap `query_tool` as the callable.

In [4]:
# NBVAL_SKIP
from typing import Optional
from pydantic import create_model
from langchain_core.tools import StructuredTool

_JSON_TO_PY = {'string': str, 'integer': int, 'number': float, 'boolean': bool}


def _schema_to_model(name, input_schema):
    # ponytail: handles flat scalar params only (all CR tool args are scalars).
    #           Nest/array support: extend _JSON_TO_PY if a future tool needs it.
    props = (input_schema or {}).get('properties', {})
    required = set((input_schema or {}).get('required', []))
    fields = {}
    for field, spec in props.items():
        py = _JSON_TO_PY.get(spec.get('type'), str)
        if field in required:
            fields[field] = (py, ...)
        else:
            fields[field] = (Optional[py], None)
    return create_model(f'{name}_Args', **fields)


def make_tool(spec):
    name = spec['name']

    async def _call(**kwargs):
        args = {k: v for k, v in kwargs.items() if v is not None}
        return str(await client.query_tool(AGENT_KEY, name, args))

    return StructuredTool.from_function(
        coroutine=_call, name=name,
        description=spec.get('description', name),
        args_schema=_schema_to_model(name, spec.get('inputSchema')))


tool_specs = await client.list_tools(AGENT_KEY)
tools = [make_tool(s) for s in tool_specs]
print(f'wrapped {len(tools)} Context Retriever tools for the agent')

wrapped 15 Context Retriever tools for the agent


## Step 3 — Build the ReAct agent

LangGraph's prebuilt ReAct agent takes the LLM and the tool list. The agent discovers what each tool does from its self-describing name and description — no hardcoded entity knowledge in the prompt.

In [5]:
# NBVAL_SKIP
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
agent = create_react_agent(llm, tools)


async def ask(question):
    result = await agent.ainvoke({'messages': [('user', question)]})
    calls = [c['name'] for m in result['messages']
             for c in getattr(m, 'tool_calls', []) or []]
    print('TOOLS USED:', calls or '(none)')
    print('ANSWER:', result['messages'][-1].content)

/var/folders/k6/xpsw19t90p5btcgd_z3hyc3w0000gp/T/ipykernel_62394/2392716563.py:6: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


## Step 4 — Ask questions

Watch which tools the agent selects.

In [6]:
# NBVAL_SKIP
await ask('What has Ada Lovelace ordered?')

TOOLS USED: ['search_customer_by_text', 'filter_order_by_customer_id']
ANSWER: Ada Lovelace has placed two orders:

1. **Order ID:** o1
   - **Summary:** Keyboard
   - **Total:** $120
   - **Status:** Shipped

2. **Order ID:** o2
   - **Summary:** Standing desk
   - **Total:** $650
   - **Status:** Pending


In [7]:
# NBVAL_SKIP
await ask('Which orders are worth more than $100, and which are still pending?')

TOOLS USED: ['find_order_by_total_range', 'filter_order_by_status', 'find_order_by_total_range']
ANSWER: Here are the orders worth more than $100:

1. **Order ID:** o2
   - **Customer ID:** c1
   - **Summary:** Standing desk
   - **Total:** $650
   - **Status:** Pending

2. **Order ID:** o1
   - **Customer ID:** c1
   - **Summary:** Keyboard
   - **Total:** $120
   - **Status:** Shipped

Among these, the only order that is still pending is the **Standing desk** (Order ID: o2).


In [8]:
# NBVAL_SKIP
await ask('Who is our highest lifetime-value customer and what did they buy?')

TOOLS USED: ['list_customer', 'filter_order_by_customer_id']
ANSWER: Our highest lifetime-value customer is **Ada Lovelace** from London, with a lifetime value of **$4200**. 

She has made the following purchases:
1. **Keyboard** - $120 (Status: Shipped)
2. **Standing desk** - $650 (Status: Pending)


## Why not just vector RAG?

The last question is the interesting one: it needs the agent to *combine* facts — find the top customer by a numeric field, then pull their orders. A single vector search can't do that reliably. The [Redis blog on context retrieval](https://redis.io/blog/context-retrieval-for-ai-agents/) lists the failure modes that show up when you push structured questions through embeddings-only RAG:

| Naive vector RAG | Context Retriever |
|---|---|
| Returns a semantically *close* text chunk — may be partial or stale | Returns **exact, complete** records |
| One-shot retrieval; the agent gets one blob and stops | Agent **chains** typed tools, narrowing each step |
| Can conflate records from different entities | Typed tools scoped per entity + field |
| You embed + re-chunk data on every change | Define the model once; tools stay in sync |

Vector RAG is still the right tool for *unstructured* knowledge (docs, transcripts, policies). Context Retriever is for the *structured, live* business records an agent must get exactly right. In production you use both — often alongside [Redis Agent Memory](https://redis.io/agent-memory/) for conversational recall.

## Cleanup

In [9]:
# NBVAL_SKIP
# Delete the surface (this also removes its agent keys) so the notebook leaves
# nothing behind in your Redis Cloud account, then close the client.
await client._api_client.delete_context_surface(surface.id, admin_key=CTX_ADMIN_KEY)
print('deleted surface:', surface.id)
await client.__aexit__(None, None, None)

deleted surface: 3a9af0ef-36a7-4cc4-b602-450214dbae24
